# Module 2.4 — Fine-Tune Encoder for Intent + Priority
**DeskMate SLM Curriculum · Phase 2 · Notebook 11**

Run on **Google Colab** (free T4 recommended, ~15 min) or CPU (~60 min).

By the end you will have:
- A fine-tuned MiniLM intent classifier (15 classes) with macro-F1 and confusion matrix
- A fine-tuned MiniLM priority classifier (3 classes)
- An inference function classifying raw ticket text in <5ms on CPU
- Both models saved to `models/`

Read `2.4_encoder_finetune.md` for the full theory.

---


## Step 0 — Install & Seed

In [ ]:
%%capture
!pip install -q transformers==4.44.0 datasets==2.21.0 torch==2.3.1 scikit-learn==1.5.1 matplotlib==3.9.0


In [ ]:
import random, json, pathlib
import torch
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_from_disk, DatasetDict, Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding,
)
from sklearn.metrics import (
    f1_score, accuracy_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay,
)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch : {torch.__version__}")
print(f"Device  : {device}")
if device == "cuda":
    print(f"GPU     : {torch.cuda.get_device_name(0)}")


## Step 1 — Runtime & Paths

In [ ]:
try:
    import google.colab; RUNTIME = "colab"
    PROJECT_ROOT = pathlib.Path("/content/slm")
except ImportError:
    try:
        import kaggle_secrets; RUNTIME = "kaggle"
        PROJECT_ROOT = pathlib.Path("/kaggle/working/slm")
    except ImportError:
        RUNTIME = "local"
        PROJECT_ROOT = pathlib.Path(".").resolve()

DATA_PROC  = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Runtime : {RUNTIME}")


## Step 2 — Label Maps & Tokenizer

In [ ]:
maps_path = DATA_PROC / "label_maps.json"
if maps_path.exists():
    label_maps  = json.loads(maps_path.read_text())
    INTENT2ID   = label_maps["INTENT2ID"]
    ID2INTENT   = {int(k): v for k, v in label_maps["ID2INTENT"].items()}
    PRIORITY2ID = label_maps["PRIORITY2ID"]
    ID2PRIORITY = {int(k): v for k, v in label_maps["ID2PRIORITY"].items()}
else:
    # Fallback — define inline
    INTENTS     = ["account_access","account_settings","billing_dispute","billing_inquiry",
                   "cancellation","data_privacy","feature_request","onboarding",
                   "outage_report","payment_failure","performance_issue","refund_request",
                   "technical_bug","usage_question","out_of_scope"]
    INTENT2ID   = {i: n for n, i in enumerate(INTENTS)}
    ID2INTENT   = {n: i for n, i in enumerate(INTENTS)}
    PRIORITIES  = ["low","medium","high"]
    PRIORITY2ID = {p: n for n, p in enumerate(PRIORITIES)}
    ID2PRIORITY = {n: p for n, p in enumerate(PRIORITIES)}

MODEL_NAME = "microsoft/MiniLM-L12-H384-uncased"
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
collator   = DataCollatorWithPadding(tokenizer=tokenizer)

N_INTENTS   = len(INTENT2ID)
N_PRIORITIES = len(PRIORITY2ID)
print(f"Intent classes  : {N_INTENTS}")
print(f"Priority classes: {N_PRIORITIES}")
print(f"Model           : {MODEL_NAME}")


## Step 3 — Load Tokenized Dataset

In [ ]:
import pandas as pd

tok_path = DATA_PROC / "tokenized"

if tok_path.exists():
    tokenized = load_from_disk(str(tok_path))
    print("Loaded tokenized dataset from disk.")
else:
    print("Tokenized dataset not found — building on-the-fly from JSONL splits.")
    # Rebuild from raw JSONL (fallback for standalone run)
    def load_raw(name):
        p = DATA_PROC / f"{name}.jsonl"
        if not p.exists():
            rows = [
                {"text": "I cannot log in.", "intent": "account_access", "priority": "medium", "source": "ph"},
                {"text": "Charged twice.", "intent": "billing_dispute", "priority": "high", "source": "ph"},
                {"text": "How do I export?", "intent": "usage_question", "priority": "low", "source": "ph"},
            ] * 20
            p.parent.mkdir(parents=True, exist_ok=True)
            p.write_text("\n".join(json.dumps(r) for r in rows))
        return Dataset.from_pandas(pd.read_json(p, lines=True))

    aug = DATA_PROC / "train_augmented.jsonl"
    raw = DatasetDict({
        "train": load_raw("train_augmented" if aug.exists() else "train"),
        "val":   load_raw("val"),
        "test":  load_raw("test"),
    })

    def tok_fn(ex):
        out = tokenizer(ex["text"], truncation=True, max_length=128, padding=False)
        out["intent_label"]   = [INTENT2ID.get(i, 0) for i in ex["intent"]]
        out["priority_label"] = [PRIORITY2ID.get(p, 0) for p in ex["priority"]]
        return out

    drop_cols = [c for c in raw["train"].column_names
                 if c not in ("input_ids","attention_mask","intent_label","priority_label")]
    tokenized = raw.map(tok_fn, batched=True, remove_columns=drop_cols)

print("Splits:")
for split, ds in tokenized.items():
    print(f"  {split:<6}: {len(ds):>6} examples")


## Step 4 — Metrics Function

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    return {
        "accuracy": float(accuracy_score(labels, preds)),
        "f1":       float(f1_score(labels, preds, average="macro", zero_division=0)),
    }


## Step 5 — Train Intent Classifier

Full fine-tune of MiniLM with a 15-class head.
3 epochs, lr=2e-5, macro-F1 as best-model metric.


In [ ]:
# Prepare dataset: rename intent_label -> labels (Trainer convention)
def rename_intent(ex):
    ex["labels"] = ex["intent_label"]
    return ex

intent_ds = tokenized.map(rename_intent, batched=True)
intent_ds.set_format("torch", columns=["input_ids","attention_mask","labels"])

intent_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=N_INTENTS,
    id2label=ID2INTENT,
    label2id=INTENT2ID,
)
n_params = sum(p.numel() for p in intent_model.parameters())
print(f"Model parameters: {n_params/1e6:.1f}M")


In [ ]:
intent_args = TrainingArguments(
    output_dir                  = str(MODELS_DIR / "intent_classifier"),
    num_train_epochs            = 3,
    per_device_train_batch_size = 32,
    per_device_eval_batch_size  = 64,
    learning_rate               = 2e-5,
    warmup_ratio                = 0.1,
    weight_decay                = 0.01,
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "f1",
    greater_is_better           = True,
    logging_steps               = 50,
    fp16                        = (device == "cuda"),
    seed                        = SEED,
    report_to                   = "none",
)

intent_trainer = Trainer(
    model           = intent_model,
    args            = intent_args,
    train_dataset   = intent_ds["train"],
    eval_dataset    = intent_ds["val"],
    data_collator   = collator,
    compute_metrics = compute_metrics,
    tokenizer       = tokenizer,
)

print("Starting intent classifier training ...")
intent_trainer.train()
print("Training complete.")


In [ ]:
# Plot training loss
logs = [x for x in intent_trainer.state.log_history if "loss" in x and "eval_loss" not in x]
eval_logs = [x for x in intent_trainer.state.log_history if "eval_f1" in x]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

if logs:
    steps = [x["step"] for x in logs]
    losses = [x["loss"] for x in logs]
    ax1.plot(steps, losses, color="#4C72B0")
    ax1.set_xlabel("Step"); ax1.set_ylabel("Train loss")
    ax1.set_title("Intent Classifier — Training Loss")

if eval_logs:
    epochs = [x["epoch"] for x in eval_logs]
    f1s    = [x["eval_f1"] for x in eval_logs]
    ax2.plot(epochs, f1s, "o-", color="#DD8452")
    ax2.set_xlabel("Epoch"); ax2.set_ylabel("Val macro-F1")
    ax2.set_title("Intent Classifier — Val F1")
    ax2.set_ylim(0, 1)

plt.tight_layout()
plt.savefig(str(MODELS_DIR / "intent_training.png"), bbox_inches="tight")
plt.show()


## Step 6 — Intent Classifier Evaluation

In [ ]:
# Evaluate on test set
test_preds = intent_trainer.predict(intent_ds["test"])
y_pred = test_preds.predictions.argmax(axis=-1)
y_true = test_preds.label_ids

print("Intent classifier — test set:")
print(f"  Accuracy  : {accuracy_score(y_true, y_pred)*100:.1f}%")
print(f"  Macro-F1  : {f1_score(y_true, y_pred, average='macro', zero_division=0)*100:.1f}%")
print()
print("Per-class F1:")
report = classification_report(y_true, y_pred,
                                target_names=[ID2INTENT[i] for i in range(N_INTENTS)],
                                zero_division=0)
print(report)


In [ ]:
# Confusion matrix
fig, ax = plt.subplots(figsize=(12, 10))
labels_short = [ID2INTENT[i][:12] for i in range(N_INTENTS)]
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=labels_short)
disp.plot(ax=ax, xticks_rotation=45, colorbar=False, cmap="Blues")
ax.set_title("Intent Classifier — Confusion Matrix (test set)")
plt.tight_layout()
plt.savefig(str(MODELS_DIR / "intent_confusion_matrix.png"), bbox_inches="tight")
plt.show()
print("Saved confusion matrix.")


## Step 7 — Train Priority Classifier

In [ ]:
def rename_priority(ex):
    ex["labels"] = ex["priority_label"]
    return ex

priority_ds = tokenized.map(rename_priority, batched=True)
priority_ds.set_format("torch", columns=["input_ids","attention_mask","labels"])

priority_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=N_PRIORITIES,
    id2label=ID2PRIORITY,
    label2id=PRIORITY2ID,
)

priority_args = TrainingArguments(
    output_dir                  = str(MODELS_DIR / "priority_classifier"),
    num_train_epochs            = 3,
    per_device_train_batch_size = 32,
    per_device_eval_batch_size  = 64,
    learning_rate               = 2e-5,
    warmup_ratio                = 0.1,
    weight_decay                = 0.01,
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "f1",
    greater_is_better           = True,
    fp16                        = (device == "cuda"),
    seed                        = SEED,
    report_to                   = "none",
)

priority_trainer = Trainer(
    model           = priority_model,
    args            = priority_args,
    train_dataset   = priority_ds["train"],
    eval_dataset    = priority_ds["val"],
    data_collator   = collator,
    compute_metrics = compute_metrics,
    tokenizer       = tokenizer,
)

print("Training priority classifier ...")
priority_trainer.train()
print("Done.")


In [ ]:
p_preds = priority_trainer.predict(priority_ds["test"])
py_pred = p_preds.predictions.argmax(axis=-1)
py_true = p_preds.label_ids

print("Priority classifier — test set:")
print(f"  Accuracy: {accuracy_score(py_true, py_pred)*100:.1f}%")
print(f"  Macro-F1: {f1_score(py_true, py_pred, average='macro', zero_division=0)*100:.1f}%")
print()
print(classification_report(py_true, py_pred,
      target_names=[ID2PRIORITY[i] for i in range(N_PRIORITIES)], zero_division=0))


## Step 8 — Inference Demo

In [ ]:
def classify_ticket(text, intent_model, priority_model, tokenizer):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128)
    intent_model.eval(); priority_model.eval()
    with torch.no_grad():
        intent_logits   = intent_model(**inputs).logits
        priority_logits = priority_model(**inputs).logits
    intent_id   = intent_logits.argmax().item()
    priority_id = priority_logits.argmax().item()
    intent_conf   = intent_logits.softmax(-1)[0, intent_id].item()
    priority_conf = priority_logits.softmax(-1)[0, priority_id].item()
    return {
        "intent":        ID2INTENT[intent_id],
        "intent_conf":   round(intent_conf, 3),
        "priority":      ID2PRIORITY[priority_id],
        "priority_conf": round(priority_conf, 3),
    }

sample_tickets = [
    "I cannot log in to my account. Password reset email never arrived.",
    "I was charged twice for the Pro plan last month. Please refund the duplicate.",
    "The export to CSV button throws a 500 error every time I click it.",
    "Our entire team has been locked out for 2 hours — we have a client call in 30 min.",
    "How do I set up the Slack integration for notifications?",
]

print(f"{'Ticket':<65} {'Intent':<22} {'Conf':>5}  {'Priority':<8} {'Conf':>5}")
print("-" * 115)
for ticket in sample_tickets:
    result = classify_ticket(ticket, intent_trainer.model, priority_trainer.model, tokenizer)
    print(f"{ticket[:64]:<65} {result['intent']:<22} {result['intent_conf']:>5.3f}  "
          f"{result['priority']:<8} {result['priority_conf']:>5.3f}")


## Step 9 — Save Models

In [ ]:
intent_save   = MODELS_DIR / "intent_classifier"
priority_save = MODELS_DIR / "priority_classifier"

intent_trainer.save_model(str(intent_save))
tokenizer.save_pretrained(str(intent_save))
priority_trainer.save_model(str(priority_save))
tokenizer.save_pretrained(str(priority_save))

print(f"Intent model saved   : {intent_save}")
print(f"Priority model saved : {priority_save}")


In [ ]:
# Optional: push to HF Hub
PUSH_TO_HUB = False

if PUSH_TO_HUB:
    from huggingface_hub import login
    try:
        from google.colab import userdata
        login(token=userdata.get("HF_TOKEN"))
    except Exception:
        import os; login(token=os.getenv("HF_TOKEN",""))

    HF_USERNAME = "YOUR_HF_USERNAME"
    intent_trainer.model.push_to_hub(f"{HF_USERNAME}/deskmate-intent-classifier")
    priority_trainer.model.push_to_hub(f"{HF_USERNAME}/deskmate-priority-classifier")
    tokenizer.push_to_hub(f"{HF_USERNAME}/deskmate-intent-classifier")
    print("Models pushed to HF Hub.")
else:
    print("Skipped Hub push (PUSH_TO_HUB=False).")


## Checkpoint

> **Your val accuracy is high but production accuracy is low — list three likely causes.**

Write your answer below.


In [ ]:
answer = """
[Write your answer here — name at least 3 causes]
"""
print(answer)


## Deliverable Summary

| Artifact | Location |
|---|---|
| Intent classifier | `models/intent_classifier/` |
| Priority classifier | `models/priority_classifier/` |
| Training loss chart | `models/intent_training.png` |
| Confusion matrix | `models/intent_confusion_matrix.png` |

**Next:** Module 2.5 — Token classification for field extraction (NER).
Using the character spans from Module 2.2's synthetic data, train
`AutoModelForTokenClassification` to pull product, version, and error_code
out of ticket text and return a structured JSON object.
